# Pengolahan dan Analisis Data Kualitas Udara

Pada tugas ini dilakukan pengolahan data kualitas udara dari tiga jenis polutan, yaitu Carbon Monoxide (CO), Nitrogen Dioxide (NO₂), dan Sulfur Dioxide (SO₂). Data awal yang digunakan masih memiliki jumlah pengamatan yang berbeda serta terdapat missing value dan outlier.

Tahapan pengolahan data dilakukan untuk mempersiapkan data sebelum digunakan pada proses ekstraksi fitur dan analisis clustering. Data yang memiliki missing value dan outlier terlebih dahulu ditangani menggunakan metode interpolasi. Dalam tugas ini digunakan dua pendekatan interpolasi, yaitu interpolasi Linear dan interpolasi Polynomial. Kedua metode tersebut diproses secara terpisah sehingga menghasilkan dua dataset hasil preprocessing yang nantinya dapat digunakan sebagai dua eksperimen.

Setelah preprocessing selesai, data dari ketiga polutan pada masing-masing metode digabungkan berdasarkan tanggal. Selanjutnya dilakukan ekstraksi fitur menggunakan TSFEL. Dari setiap polutan diekstraksi sebanyak 68 fitur sehingga dari tiga polutan diperoleh:

68 fitur × 3 polutan = 204 fitur.

Hasil ekstraksi fitur dari metode Linear dan Polynomial selanjutnya digunakan sebagai dasar untuk tahap analisis berikutnya, yaitu reduksi dimensi, clustering, evaluasi menggunakan Silhouette Coefficient, dan visualisasi hasil clustering pada peta.

## Import Library

Tahap pertama adalah mempersiapkan library yang dibutuhkan selama proses pengolahan dan analisis data. Library yang digunakan mencakup library untuk membaca dan mengolah data, melakukan perhitungan numerik, visualisasi, reduksi dimensi, clustering, serta evaluasi hasil clustering.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import warnings
warnings.filterwarnings('ignore')

## 2. Membaca dan Melihat Struktur Data

Data awal terdiri dari tiga dataset kualitas udara, yaitu CO, NO₂, dan SO₂. Ketiga data tersebut masih disimpan dalam file yang terpisah. Oleh karena itu, masing-masing file perlu dibaca terlebih dahulu untuk mengetahui struktur dan jumlah data yang tersedia.

Sebelum melakukan preprocessing, struktur data perlu diperiksa terlebih dahulu. Pemeriksaan dilakukan untuk mengetahui contoh isi data serta memastikan nama kolom yang digunakan sudah sesuai.

In [85]:
co = pd.read_csv(r'..\data\csv\CO-Kedungpring.csv')
no2 = pd.read_csv(r'..\data\csv\NO2-Kedungpring.csv')
so2 = pd.read_csv(r'..\data\csv\SO2-Kedungpring.csv')

print("CO :", co.shape)
print("NO2:", no2.shape)
print("SO2:", so2.shape)

display(co.head())
display(no2.head())
display(so2.head())

print("Kolom CO :", co.columns.tolist())
print("Kolom NO2:", no2.columns.tolist())
print("Kolom SO2:", so2.columns.tolist())


CO : (205, 2)
NO2: (169, 2)
SO2: (221, 2)


,date,CO
0,2025-09-01,0.028042
1,2025-09-02,0.026509
2,2025-09-03,0.025827
3,2025-09-04,0.032963
4,2025-09-06,0.029518


,date,NO2
0,2025-08-31,0.000036
1,2025-09-01,0.000020
2,2025-09-03,0.000013
3,2025-09-05,0.000029
4,2025-09-11,0.000030


,date,SO2
0,2025-08-31,0.000066
1,2025-09-01,0.000237
2,2025-09-03,-0.000250
3,2025-09-04,0.000002
4,2025-09-05,-0.000122


Kolom CO : ['date', 'CO']
Kolom NO2: ['date', 'NO2']
Kolom SO2: ['date', 'SO2']


Hasil pembacaan menunjukkan bahwa jumlah baris pada masing-masing dataset belum sama. Hal ini menunjukkan bahwa data ketiga polutan belum berada pada rentang pengamatan yang seragam sehingga diperlukan proses penyamaan tanggal pada tahap berikutnya.

Dari pemeriksaan Struktur data diketahui bahwa setiap dataset memiliki kolom date sebagai tanggal pengamatan dan satu kolom yang berisi nilai polutan masing-masing, yaitu CO, NO2, dan SO2.


## 3.Persiapan dan Penyamaan Data berdasarkan Tanggal

Sebelum melakukan preprocessing, seluruh data polutan perlu dipersiapkan agar memiliki format tanggal yang seragam dan rentang pengamatan yang sama. Kolom date pada data CO, NO₂, dan SO₂ terlebih dahulu diubah menjadi format datetime. Setelah itu, data disusun berdasarkan tanggal dan dibuat rentang tanggal lengkap dari 31 Agustus 2025 sampai 31 Agustus 2026, sehingga diperoleh 366 tanggal pengamatan. Selanjutnya, masing-masing dataset disesuaikan dengan rentang tanggal tersebut menggunakan reindex(). Proses ini membuat ketiga polutan memiliki jumlah baris dan tanggal yang sama. Tanggal yang sebelumnya tidak memiliki data akan menghasilkan nilai NaN dan nantinya ditangani pada tahap interpolasi.

In [33]:
co['date'] = pd.to_datetime(co['date'])
no2['date'] = pd.to_datetime(no2['date'])
so2['date'] = pd.to_datetime(so2['date'])

print(co['date'].dtype)
print(no2['date'].dtype)
print(so2['date'].dtype)

datetime64[ns]
datetime64[ns]
datetime64[ns]


In [48]:
tanggal_lengkap = pd.date_range(
    start='2025-08-31',
    end='2026-08-31',
    freq='D'
)

print("Jumlah tanggal:", len(tanggal_lengkap))

Jumlah tanggal: 366


In [50]:
co_365 = co_clean.copy()

co_365['date'] = pd.to_datetime(co_365['date'])

co_365 = co_365.set_index('date')

co_365 = co_365.reindex(tanggal_lengkap)

co_365.index.name = 'date'

co_365 = co_365.reset_index()

print("Jumlah baris CO:", len(co_365))

no2_365 = no2_clean.copy()

no2_365['date'] = pd.to_datetime(no2_365['date'])

no2_365 = no2_365.set_index('date')
no2_365 = no2_365.reindex(tanggal_lengkap)

no2_365.index.name = 'date'
no2_365 = no2_365.reset_index()

print("Jumlah baris NO2:", len(no2_365))

so2_365 = so2_clean.copy()

so2_365['date'] = pd.to_datetime(so2_365['date'])

so2_365 = so2_365.set_index('date')
so2_365 = so2_365.reindex(tanggal_lengkap)

so2_365.index.name = 'date'
so2_365 = so2_365.reset_index()

print("Jumlah baris SO2:", len(so2_365))

Jumlah baris CO: 366
Jumlah baris NO2: 366
Jumlah baris SO2: 366


Setelah proses ini, ketiga dataset memiliki 366 baris dengan rentang tanggal yang sama. Nilai NaN yang muncul berasal dari tanggal yang tidak memiliki data pengukuran dan akan diproses pada tahap interpolasi Linear dan Polynomial.

## 4. Pengecekan Missing Value

Setelah proses ini, ketiga dataset memiliki 366 baris dengan rentang tanggal yang sama. Nilai NaN yang muncul berasal dari tanggal yang tidak memiliki data pengukuran dan akan diproses pada tahap interpolasi Linear dan Polynomial.

In [51]:
print("CO :", co_365['CO'].isna().sum())
print("NO2:", no2_365['NO2'].isna().sum())
print("SO2:", so2_365['SO2'].isna().sum())

CO : 162
NO2: 197
SO2: 157


Hasil pengecekan menunjukkan bahwa masih terdapat nilai NaN pada ketiga polutan. Nilai tersebut tidak langsung dihapus karena penghapusan data akan mengurangi jumlah pengamatan. Oleh karena itu, missing value ditangani menggunakan metode interpolasi

## 5. Preprocessing dengan Interpolasi Linear

Pada eksperimen pertama digunakan interpolasi Linear untuk memperkirakan nilai yang hilang. Metode ini memperkirakan nilai berdasarkan hubungan linear antara nilai yang tersedia di sekitar posisi missing value.

Hasil preprocessing Linear disimpan secara terpisah agar dapat digunakan sebagai dataset eksperimen pertama.

Setelah interpolasi Linear, nilai yang kosong diestimasi berdasarkan pola nilai yang tersedia. Dataset ini menjadi hasil preprocessing Linear dan akan digunakan sebagai eksperimen pertama.

In [ ]:

# INTERPOLASI LINEAR

co_linear = co_365.copy()
no2_linear = no2_365.copy()
so2_linear = so2_365.copy()

co_linear = co_linear.set_index('date')
no2_linear = no2_linear.set_index('date')
so2_linear = so2_linear.set_index('date')

co_linear['CO'] = co_linear['CO'].interpolate(
    method='linear',
    limit_direction='both'
)

no2_linear['NO2'] = no2_linear['NO2'].interpolate(
    method='linear',
    limit_direction='both'
)

so2_linear['SO2'] = so2_linear['SO2'].interpolate(
    method='linear',
    limit_direction='both'
)

co_linear = co_linear.reset_index()
no2_linear = no2_linear.reset_index()
so2_linear = so2_linear.reset_index()

## 6. Preprocessing dengan Interpolasi Polynomial

Setelah interpolasi Linear, nilai yang kosong diestimasi berdasarkan pola nilai yang tersedia. Dataset ini menjadi hasil preprocessing Linear dan akan digunakan sebagai eksperimen pertama.

Hasilnya menjadi dataset preprocessing Polynomial. Dataset ini tetap dipisahkan dari hasil Linear karena nantinya kedua metode akan menjadi dua eksperimen yang dibandingkan.

In [ ]:
# INTERPOLASI POLYNOMIAL

co_polynomial = co_365.copy()
no2_polynomial = no2_365.copy()
so2_polynomial = so2_365.copy()

co_polynomial = co_polynomial.set_index('date')
no2_polynomial = no2_polynomial.set_index('date')
so2_polynomial = so2_polynomial.set_index('date')

co_polynomial['CO'] = co_polynomial['CO'].interpolate(
    method='polynomial',
    order=2,
    limit_direction='both'
)

no2_polynomial['NO2'] = no2_polynomial['NO2'].interpolate(
    method='polynomial',
    order=2,
    limit_direction='both'
)

so2_polynomial['SO2'] = so2_polynomial['SO2'].interpolate(
    method='polynomial',
    order=2,
    limit_direction='both'
)

co_polynomial = co_polynomial.reset_index()
no2_polynomial = no2_polynomial.reset_index()
so2_polynomial = so2_polynomial.reset_index()

## 7. Menangin NaN pada Ujung Data

Pada kondisi tertentu, interpolasi Polynomial masih dapat meninggalkan nilai NaN, terutama apabila missing value berada pada bagian paling awal atau paling akhir data. Oleh karena itu digunakan fungsi tambahan untuk memperkirakan nilai tersebut menggunakan beberapa data valid terdekat dengan model Polynomial orde 2.

In [72]:
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

def isi_nan_ujung_polynomial(data, kolom, jumlah_data=5, order=2):
    data = data.copy()

    indeks_nan = data[data[kolom].isna()].index

    for idx in indeks_nan:

        # =========================
        # NaN berada di awal data
        # =========================
        if idx == data.index.min():

            data_valid = data.loc[idx+1:, kolom].dropna().head(jumlah_data)

            if len(data_valid) >= order + 1:

                X = np.arange(len(data_valid)).reshape(-1, 1)
                y = data_valid.values

                poly = PolynomialFeatures(degree=order)
                X_poly = poly.fit_transform(X)

                model = LinearRegression()
                model.fit(X_poly, y)

                # Prediksi posisi sebelum data pertama
                X_pred = np.array([[-1]])
                X_pred_poly = poly.transform(X_pred)

                hasil = model.predict(X_pred_poly)[0]

                data.loc[idx, kolom] = hasil

        # =========================
        # NaN berada di akhir data
        # =========================
        else:

            data_valid = data.loc[:idx-1, kolom].dropna().tail(jumlah_data)

            if len(data_valid) >= order + 1:

                X = np.arange(len(data_valid)).reshape(-1, 1)
                y = data_valid.values

                poly = PolynomialFeatures(degree=order)
                X_poly = poly.fit_transform(X)

                model = LinearRegression()
                model.fit(X_poly, y)

                # Prediksi posisi setelah data terakhir
                X_pred = np.array([[len(data_valid)]])
                X_pred_poly = poly.transform(X_pred)

                hasil = model.predict(X_pred_poly)[0]

                data.loc[idx, kolom] = hasil

    return data

In [73]:
co_polynomial = isi_nan_ujung_polynomial(
    co_polynomial,
    'CO'
)

no2_polynomial = isi_nan_ujung_polynomial(
    no2_polynomial,
    'NO2'
)

so2_polynomial = isi_nan_ujung_polynomial(
    so2_polynomial,
    'SO2'
)

## 8. Validasi Hasil Preprocessing

Setelah kedua metode interpolasi selesai, dilakukan pengecekan untuk memastikan bahwa jumlah data tetap 366 baris dan tidak terdapat missing value.

In [76]:
print("=== JUMLAH BARIS ===")
print("CO :", len(co_linear))
print("NO2:", len(no2_linear))
print("SO2:", len(so2_linear))

print("\n=== LINEAR ===")
print("CO :", co_linear['CO'].isna().sum())
print("NO2:", no2_linear['NO2'].isna().sum())
print("SO2:", so2_linear['SO2'].isna().sum())

print("\n=== POLYNOMIAL ===")
print("CO :", co_polynomial['CO'].isna().sum())
print("NO2:", no2_polynomial['NO2'].isna().sum())
print("SO2:", so2_polynomial['SO2'].isna().sum())



=== JUMLAH BARIS ===
CO : 366
NO2: 366
SO2: 366

=== LINEAR ===
CO : 0
NO2: 0
SO2: 0

=== POLYNOMIAL ===
CO : 0
NO2: 0
SO2: 0


## 9. Menggabungkan Tiga Polutan dan Menyimpan Hasil Preprocessing

Setelah preprocessing selesai, ketiga polutan digabungkan berdasarkan kolom date. Penggabungan dilakukan di masing-masing metode, sehingga tidak mencampurkan hasil Linear dengan Polynomial.

Dengan demikian terdapat dua dataset:

- Dataset gabungan hasil Linear
- Dataset gabungan hasil Polynomial

Kedua dataset hasil preprocessing disimpan sebagai file CSV agar dapat digunakan kembali pada proses ekstraksi fitur.

In [80]:
data_linear = co_linear[['date', 'CO']].merge(
    no2_linear[['date', 'NO2']],
    on='date'
).merge(
    so2_linear[['date', 'SO2']],
    on='date'
)

data_polynomial = co_polynomial[['date', 'CO']].merge(
    no2_polynomial[['date', 'NO2']],
    on='date'
).merge(
    so2_polynomial[['date', 'SO2']],
    on='date'
)

print("Linear:", data_linear.shape)
print("Polynomial:", data_polynomial.shape)

data_linear.to_csv(
    '../data/csv/Hasil_Preprocessing_Linear.csv',
    index=False
)

data_polynomial.to_csv(
    '../data/csv/Hasil_Preprocessing_Polynomial.csv',
    index=False
)

Linear: (366, 4)
Polynomial: (366, 4)


## 11. Ekstraksi Fitur TSFEL

Setelah preprocessing selesai, tahap berikutnya adalah ekstraksi fitur. Pada tahap ini digunakan library TSFEL (Time Series Feature Extraction Library) untuk mengambil karakteristik statistik dan karakteristik sinyal dari data polutan.

Pada tugas ini digunakan 68 fitur untuk setiap polutan. Karena terdapat tiga polutaN (CO,NO2,SO2) maka Jumlah Keseluruhan fitur 204.

Proses ini dilakukan 2 kali dengan menggunakan hasil preprocessing linear dan hasil preprocessing polynomial, Kedua dataset hasil ekstraksi fitur disimpan sebagai file CSV agar dapat digunakan kembali pada proses Clustering.


In [83]:
import pandas as pd
import numpy as np
import inspect
import tsfel.feature_extraction.features as tsfel_features

# ---------- 1. Muat 1 file CSV utama yang berisi semua polutan ----------
# Sesuaikan nama file jika berbeda (misal: 'mystorage/Kerek-Clean.csv')
df = pd.read_csv('../data/csv/Hasil_Preprocessing_Polynomial.csv')

df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

pollutants = ['NO2', 'SO2', 'CO']
fs = 1

# ---------- 2. Daftar PERSIS 68 fitur yang diminta ----------
FEATURE_LIST = """abs_energy auc autocorr average_power calc_centroid calc_max calc_mean
calc_median calc_min calc_std calc_var dfa distance ecdf ecdf_percentile ecdf_percentile_count
ecdf_slope entropy fundamental_frequency higuchi_fractal_dimension hist_mode human_range_energy
hurst_exponent interq_range kurtosis lempel_ziv lpcc max_frequency max_power_spectrum
maximum_fractal_length mean_abs_deviation mean_abs_diff mean_diff median_abs_deviation
median_abs_diff median_diff median_frequency mfcc mse negative_turning neighbourhood_peaks
petrosian_fractal_dimension pk_pk_distance positive_turning power_bandwidth rms skewness slope
spectral_centroid spectral_decrease spectral_distance spectral_entropy spectral_kurtosis
spectral_positive_turning spectral_roll_off spectral_roll_on spectral_skewness spectral_slope
spectral_spread spectral_variation spectrogram_mean_coeff sum_abs_diff wavelet_abs_mean
wavelet_energy wavelet_entropy wavelet_std wavelet_var zero_cross""".split()

print(f"Jumlah fitur per polutan: {len(FEATURE_LIST)}")
print(f"Total target fitur keseluruhan: {len(FEATURE_LIST) * len(pollutants)}")


def to_scalar(result):
    if isinstance(result, dict) and "values" in result:
        result = result["values"]
    if isinstance(result, (list, tuple, np.ndarray)):
        arr = np.asarray(result, dtype=float)
        return float(np.nanmean(arr))
    return float(result)


def extract_one(fn_name, signal, fs):
    fn = getattr(tsfel_features, fn_name)
    params = inspect.signature(fn).parameters
    if "fs" in params:
        result = fn(signal, fs)
    else:
        result = fn(signal)
    return to_scalar(result)


# Dictionary untuk menampung seluruh hasil ekstraksi dari semua polutan
combined_row = {}

# ---------- 3. Looping untuk membersihkan dan mengekstraksi tiap polutan ----------
for pollutant in pollutants:
    print(f"\n--- Memproses polutan: {pollutant} ---")
    
    # Buat copy dataframe agar tidak saling menimpa
    df_poly = df[['date', pollutant]].copy()

    # Paksa kolom target jadi numerik
    df_poly[pollutant] = pd.to_numeric(df_poly[pollutant], errors='coerce')
    n_missing_before = df_poly[pollutant].isna().sum()
    print(f"Jumlah NaN/non-numerik pada {pollutant}: {n_missing_before}")

    # Handling outlier dengan IQR
    Q1 = df_poly[pollutant].quantile(0.25)
    Q3 = df_poly[pollutant].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df_poly.loc[(df_poly[pollutant] < lower_bound) | (df_poly[pollutant] > upper_bound), pollutant] = np.nan

    # Interpolasi waktu dan cleaning
    df_clean = df_poly.set_index('date').interpolate(method='time').ffill().bfill()
    signal_1d = df_clean[pollutant].astype(float).values

    # Ekstraksi fitur dan beri prefix nama polutan (misal: NO2_abs_energy)
    for fn_name in FEATURE_LIST:
        feature_key = f"{pollutant}_{fn_name}"
        combined_row[feature_key] = extract_one(fn_name, signal_1d, fs)

# ---------- 4. Simpan ke DataFrame final (1 baris, 204 kolom) ----------
extracted_features_final = pd.DataFrame([combined_row])

print(f"\nBerhasil! Total kolom akhir yang dihasilkan: {extracted_features_final.shape[1]}")

output_filename = '../data/features/Pollutants-Kedungpring-Polynomial-TSFEL.csv'
extracted_features_final.to_csv(output_filename, index=False)
print(f"File berhasil disimpan sebagai: {output_filename}")

Jumlah fitur per polutan: 68
Total target fitur keseluruhan: 204

--- Memproses polutan: NO2 ---
Jumlah NaN/non-numerik pada NO2: 0

--- Memproses polutan: SO2 ---
Jumlah NaN/non-numerik pada SO2: 0

--- Memproses polutan: CO ---
Jumlah NaN/non-numerik pada CO: 0

Berhasil! Total kolom akhir yang dihasilkan: 204
File berhasil disimpan sebagai: ../data/features/Pollutants-Kedungpring-Polynomial-TSFEL.csv


In [84]:
import pandas as pd
import numpy as np
import inspect
import tsfel.feature_extraction.features as tsfel_features

# ---------- 1. Muat 1 file CSV utama yang berisi semua polutan ----------
# Sesuaikan nama file jika berbeda (misal: 'mystorage/Kerek-Clean.csv')
df = pd.read_csv('../data/csv/Hasil_Preprocessing_Linear.csv')

df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

pollutants = ['NO2', 'SO2', 'CO']
fs = 1

# ---------- 2. Daftar PERSIS 68 fitur yang diminta ----------
FEATURE_LIST = """abs_energy auc autocorr average_power calc_centroid calc_max calc_mean
calc_median calc_min calc_std calc_var dfa distance ecdf ecdf_percentile ecdf_percentile_count
ecdf_slope entropy fundamental_frequency higuchi_fractal_dimension hist_mode human_range_energy
hurst_exponent interq_range kurtosis lempel_ziv lpcc max_frequency max_power_spectrum
maximum_fractal_length mean_abs_deviation mean_abs_diff mean_diff median_abs_deviation
median_abs_diff median_diff median_frequency mfcc mse negative_turning neighbourhood_peaks
petrosian_fractal_dimension pk_pk_distance positive_turning power_bandwidth rms skewness slope
spectral_centroid spectral_decrease spectral_distance spectral_entropy spectral_kurtosis
spectral_positive_turning spectral_roll_off spectral_roll_on spectral_skewness spectral_slope
spectral_spread spectral_variation spectrogram_mean_coeff sum_abs_diff wavelet_abs_mean
wavelet_energy wavelet_entropy wavelet_std wavelet_var zero_cross""".split()

print(f"Jumlah fitur per polutan: {len(FEATURE_LIST)}")
print(f"Total target fitur keseluruhan: {len(FEATURE_LIST) * len(pollutants)}")


def to_scalar(result):
    if isinstance(result, dict) and "values" in result:
        result = result["values"]
    if isinstance(result, (list, tuple, np.ndarray)):
        arr = np.asarray(result, dtype=float)
        return float(np.nanmean(arr))
    return float(result)


def extract_one(fn_name, signal, fs):
    fn = getattr(tsfel_features, fn_name)
    params = inspect.signature(fn).parameters
    if "fs" in params:
        result = fn(signal, fs)
    else:
        result = fn(signal)
    return to_scalar(result)


# Dictionary untuk menampung seluruh hasil ekstraksi dari semua polutan
combined_row = {}

# ---------- 3. Looping untuk membersihkan dan mengekstraksi tiap polutan ----------
for pollutant in pollutants:
    print(f"\n--- Memproses polutan: {pollutant} ---")
    
    # Buat copy dataframe agar tidak saling menimpa
    df_poly = df[['date', pollutant]].copy()

    # Paksa kolom target jadi numerik
    df_poly[pollutant] = pd.to_numeric(df_poly[pollutant], errors='coerce')
    n_missing_before = df_poly[pollutant].isna().sum()
    print(f"Jumlah NaN/non-numerik pada {pollutant}: {n_missing_before}")

    # Handling outlier dengan IQR
    Q1 = df_poly[pollutant].quantile(0.25)
    Q3 = df_poly[pollutant].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df_poly.loc[(df_poly[pollutant] < lower_bound) | (df_poly[pollutant] > upper_bound), pollutant] = np.nan

    # Interpolasi waktu dan cleaning
    df_clean = df_poly.set_index('date').interpolate(method='time').ffill().bfill()
    signal_1d = df_clean[pollutant].astype(float).values

    # Ekstraksi fitur dan beri prefix nama polutan (misal: NO2_abs_energy)
    for fn_name in FEATURE_LIST:
        feature_key = f"{pollutant}_{fn_name}"
        combined_row[feature_key] = extract_one(fn_name, signal_1d, fs)

# ---------- 4. Simpan ke DataFrame final (1 baris, 204 kolom) ----------
extracted_features_final = pd.DataFrame([combined_row])

print(f"\nBerhasil! Total kolom akhir yang dihasilkan: {extracted_features_final.shape[1]}")

output_filename = '../data/features/Pollutants-Kedungpring-Linear-TSFEL.csv'
extracted_features_final.to_csv(output_filename, index=False)
print(f"File berhasil disimpan sebagai: {output_filename}")

Jumlah fitur per polutan: 68
Total target fitur keseluruhan: 204

--- Memproses polutan: NO2 ---
Jumlah NaN/non-numerik pada NO2: 0

--- Memproses polutan: SO2 ---
Jumlah NaN/non-numerik pada SO2: 0

--- Memproses polutan: CO ---
Jumlah NaN/non-numerik pada CO: 0

Berhasil! Total kolom akhir yang dihasilkan: 204
File berhasil disimpan sebagai: ../data/features/Pollutants-Kedungpring-Linear-TSFEL.csv


# Kesimpulan

Berdasarkan proses pengolahan data kualitas udara di wilayah Kedungpring, data konsentrasi CO, NO₂, dan SO₂ berhasil disiapkan dalam rentang 31 Agustus 2025 sampai 31 Agustus 2026 sehingga seluruh polutan memiliki 366 data harian yang seragam. Data yang memiliki nilai hilang akibat proses penyamaan tanggal kemudian ditangani menggunakan dua metode interpolasi, yaitu interpolasi linear dan interpolasi polynomial orde 2. Hasil validasi menunjukkan bahwa kedua metode berhasil menghasilkan data lengkap tanpa nilai NaN pada ketiga polutan.

Selanjutnya, data hasil interpolasi digabungkan berdasarkan tanggal dan digunakan sebagai input untuk ekstraksi fitur menggunakan TSFEL. Setiap polutan menghasilkan 68 fitur, sehingga diperoleh 204 fitur untuk masing-masing metode interpolasi. Dengan demikian, terdapat dua dataset fitur yang terpisah, yaitu dataset hasil interpolasi linear dan dataset hasil interpolasi polynomial, yang selanjutnya dapat digunakan untuk tahap reduksi dimensi, clustering, dan evaluasi silhouette.